In [12]:
# ============================================================
# DAY 5 — TECHNICAL ANALYSIS (CLEAN VERSION)
# AI Stock Screener — Indian Markets
# ============================================================
 
# ── CELL 1: IMPORTS AND LOAD DATA ──────────────────────────
import pandas as pd
import numpy as np
import yfinance as yf
import pickle
import warnings
warnings.filterwarnings('ignore')
 
# Load fundamental scores
fundamental_df = pd.read_csv('../data/fundamental_scores.csv')
print(f"Fundamental picks loaded: {len(fundamental_df)} stocks")
print(f"\nTop 10:")
for i, row in fundamental_df.sort_values(
    'Final Score', ascending=False
).head(10).reset_index(drop=True).iterrows():
    print(f"  {i+1:2}. {row['Symbol']:15} {row['Final Score']}/100")

Fundamental picks loaded: 35 stocks

Top 10:
   1. PERSISTENT      80.0/100
   2. MUTHOOTFIN      73.4/100
   3. FINEORG         73.0/100
   4. CHOLAFIN        70.7/100
   5. DHANUKA         69.0/100
   6. TITAN           69.0/100
   7. BEL             68.0/100
   8. PIIND           68.0/100
   9. ABSLAMC         67.0/100
  10. SUNPHARMA       65.0/100


In [13]:
# ── CELL 2: FETCH PRICE DATA ────────────────────────────────
symbols    = fundamental_df['Symbol'].tolist()
price_data = {}
failed     = []
 
print(f"Fetching maximum price history for {len(symbols)} stocks...")
 
for symbol in symbols:
    try:
        ticker = yf.Ticker(f"{symbol}.NS")
        hist   = ticker.history(period="max")
        if len(hist) > 100:
            price_data[symbol] = hist
            start = hist.index[0].strftime('%Y')
            end   = hist.index[-1].strftime('%Y')
            print(f"  {symbol:15} {len(hist)} days ({start}-{end})")
        else:
            failed.append(symbol)
    except Exception as e:
        failed.append(symbol)
        print(f"  FAILED: {symbol} — {e}")
 
print(f"\nFetched : {len(price_data)} stocks")
print(f"Failed  : {len(failed)} — {failed}")
 
# Save
with open('../data/price_data.pkl', 'wb') as f:
    pickle.dump(price_data, f)
print("Saved price_data.pkl")

Fetching maximum price history for 35 stocks...
  TCS             5856 days (2002-2026)
  RELIANCE        7580 days (1996-2026)
  WIPRO           7583 days (1996-2026)
  PERSISTENT      3938 days (2010-2026)
  LTTS            2341 days (2016-2026)
  HAPPSTMNDS      1359 days (2020-2026)
  CHOLAFIN        5885 days (2002-2026)
  MUTHOOTFIN      3665 days (2011-2026)
  MANAPPURAM      3878 days (2010-2026)
  CREDITACC       1866 days (2018-2026)
  GARFIBRES       5857 days (2002-2026)
  SUPRAJIT        5201 days (2005-2026)
  HIMATSEIDE      5885 days (2002-2026)
  HAL             1966 days (2018-2026)
  BEL             5885 days (2002-2026)
  MIDHANI         1964 days (2018-2026)
  PARAS           1101 days (2021-2026)
  VINATIORGA      4106 days (2009-2026)
  FINEORG         1902 days (2018-2026)
  GALAXYSURF      1998 days (2018-2026)
  AAVAS           1837 days (2018-2026)
  SUNPHARMA       7583 days (1996-2026)
  DIVISLAB        5706 days (2003-2026)
  APLLTD          3571 days (201

In [14]:
# ── CELL 3: LOAD PRICE DATA (if already fetched) ────────────
with open('../data/price_data.pkl', 'rb') as f:
    price_data = pickle.load(f)
print(f"Price data loaded: {len(price_data)} stocks")

Price data loaded: 35 stocks


In [15]:
# ── CELL 4: CALCULATE INDICATORS ───────────────────────────
def calculate_indicators(df):
    data = df.copy()
 
    # EMA
    data['EMA20']  = data['Close'].ewm(span=20,  adjust=False).mean()
    data['EMA50']  = data['Close'].ewm(span=50,  adjust=False).mean()
    data['EMA200'] = data['Close'].ewm(span=200, adjust=False).mean()
 
    # RSI
    delta    = data['Close'].diff()
    gain     = delta.where(delta > 0, 0)
    loss     = -delta.where(delta < 0, 0)
    avg_gain = gain.ewm(span=14, adjust=False).mean()
    avg_loss = loss.ewm(span=14, adjust=False).mean()
    rs       = avg_gain / avg_loss
    data['RSI'] = 100 - (100 / (1 + rs))
 
    # MACD
    ema12          = data['Close'].ewm(span=12, adjust=False).mean()
    ema26          = data['Close'].ewm(span=26, adjust=False).mean()
    data['MACD']   = ema12 - ema26
    data['Signal'] = data['MACD'].ewm(span=9, adjust=False).mean()
    data['MACD_Hist'] = data['MACD'] - data['Signal']
 
    # ADX
    high     = data['High']
    low      = data['Low']
    close    = data['Close']
    tr1      = high - low
    tr2      = (high - close.shift()).abs()
    tr3      = (low  - close.shift()).abs()
    tr       = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    dm_plus  = high.diff()
    dm_minus = -low.diff()
    dm_plus  = dm_plus.where((dm_plus > dm_minus) & (dm_plus > 0), 0)
    dm_minus = dm_minus.where((dm_minus > dm_plus) & (dm_minus > 0), 0)
    atr      = tr.ewm(span=14, adjust=False).mean()
    di_plus  = 100 * dm_plus.ewm(span=14, adjust=False).mean() / atr
    di_minus = 100 * dm_minus.ewm(span=14, adjust=False).mean() / atr
    dx       = 100 * (di_plus - di_minus).abs() / (di_plus + di_minus)
    data['ADX']      = dx.ewm(span=14, adjust=False).mean()
    data['DI_Plus']  = di_plus
    data['DI_Minus'] = di_minus
 
    # Volume
    data['Vol_MA20']  = data['Volume'].rolling(20).mean()
    data['Vol_Ratio'] = data['Volume'] / data['Vol_MA20']
 
    return data
 
 
# Calculate for all stocks
indicator_data = {}
for symbol, df in price_data.items():
    try:
        indicator_data[symbol] = calculate_indicators(df)
    except Exception as e:
        print(f"FAILED: {symbol} — {e}")
 
print(f"Indicators calculated: {len(indicator_data)} stocks")
 
# Save
with open('../data/indicator_data.pkl', 'wb') as f:
    pickle.dump(indicator_data, f)
print("Saved indicator_data.pkl")
 
 
# ── CELL 5: LOAD INDICATOR DATA (if already calculated) ────
with open('../data/indicator_data.pkl', 'rb') as f:
    indicator_data = pickle.load(f)
print(f"Indicator data loaded: {len(indicator_data)} stocks")

Indicators calculated: 35 stocks
Saved indicator_data.pkl
Indicator data loaded: 35 stocks


In [16]:
# ── CELL 6: VOLUME PROFILE ──────────────────────────────────
def calculate_volume_profile(df, lookback_days=252, bins=30):
    recent      = df.tail(lookback_days).copy()
    price_min   = recent['Low'].min()
    price_max   = recent['High'].max()
 
    if price_max <= price_min:
        return None
 
    price_bins      = np.linspace(price_min, price_max, bins + 1)
    volume_at_price = np.zeros(bins)
 
    for _, row in recent.iterrows():
        candle_low   = row['Low']
        candle_high  = row['High']
        candle_vol   = row['Volume']
        candle_range = candle_high - candle_low
        if candle_range == 0:
            continue
        for i in range(bins):
            bin_low      = price_bins[i]
            bin_high     = price_bins[i + 1]
            overlap_low  = max(candle_low,  bin_low)
            overlap_high = min(candle_high, bin_high)
            if overlap_high > overlap_low:
                overlap_pct          = (overlap_high - overlap_low) / candle_range
                volume_at_price[i]  += candle_vol * overlap_pct
 
    poc_idx   = np.argmax(volume_at_price)
    poc_price = (price_bins[poc_idx] + price_bins[poc_idx + 1]) / 2
 
    total_vol  = volume_at_price.sum()
    target_vol = total_vol * 0.70
    lower_idx  = poc_idx
    upper_idx  = poc_idx
    va_vol     = volume_at_price[poc_idx]
 
    while va_vol < target_vol:
        can_up   = upper_idx + 1 < bins
        can_down = lower_idx - 1 >= 0
        exp_up   = volume_at_price[upper_idx + 1] if can_up   else 0
        exp_down = volume_at_price[lower_idx - 1] if can_down else 0
        if not can_up and not can_down:
            break
        if exp_up >= exp_down and can_up:
            upper_idx += 1
            va_vol    += exp_up
        elif can_down:
            lower_idx -= 1
            va_vol    += exp_down
        else:
            break
 
    val      = (price_bins[lower_idx] + price_bins[lower_idx + 1]) / 2
    vah      = (price_bins[upper_idx] + price_bins[upper_idx + 1]) / 2
    max_vol  = volume_at_price.max()
    edge_vol = (volume_at_price[:3].mean() + volume_at_price[-3:].mean()) / 2
    is_bell  = max_vol > edge_vol * 2.5
 
    current_price = df['Close'].iloc[-1]
 
    if current_price > vah * 1.02:
        breakout_status = 'BREAKOUT_UP'
    elif current_price < val * 0.98:
        breakout_status = 'BREAKOUT_DOWN'
    elif current_price > vah * 0.98:
        breakout_status = 'AT_RESISTANCE'
    else:
        breakout_status = 'INSIDE'
 
    return {
        'poc_price'      : round(poc_price, 2),
        'val'            : round(val,       2),
        'vah'            : round(vah,       2),
        'is_bell'        : is_bell,
        'breakout_status': breakout_status,
        'current_price'  : round(current_price, 2),
        'price_vs_poc'   : round((current_price - poc_price) / poc_price * 100, 2),
        'price_vs_vah'   : round((current_price - vah) / vah * 100, 2),
    }
 

In [17]:
# ── CELL 7: MOMENTUM SCORER ────────────────────────────────
def score_momentum(df, vp):
    scores = {}
    latest = df.iloc[-1]
    close  = latest['Close']
 
    # EMA alignment (30 pts)
    ema20  = latest['EMA20']
    ema50  = latest['EMA50']
    ema200 = latest['EMA200']
    if close > ema20 > ema50 > ema200:  scores['EMA'] = 30
    elif close > ema50 > ema200:        scores['EMA'] = 20
    elif close > ema200:                scores['EMA'] = 10
    else:                               scores['EMA'] = 0
 
    # RSI (20 pts)
    rsi = latest['RSI']
    if 50 <= rsi <= 70:     scores['RSI'] = 20
    elif 45 <= rsi < 50:    scores['RSI'] = 12
    elif 70 < rsi <= 80:    scores['RSI'] = 8
    else:                   scores['RSI'] = 0
 
    # MACD (20 pts)
    macd   = latest['MACD']
    signal = latest['Signal']
    hist   = latest['MACD_Hist']
    if macd > signal and hist > 0 and macd > 0:  scores['MACD'] = 20
    elif macd > signal and hist > 0:             scores['MACD'] = 15
    elif macd > signal:                          scores['MACD'] = 10
    else:                                        scores['MACD'] = 0
 
    # ADX (15 pts)
    adx      = latest['ADX']
    di_plus  = latest['DI_Plus']
    di_minus = latest['DI_Minus']
    if adx > 25 and di_plus > di_minus:   scores['ADX'] = 15
    elif adx > 20 and di_plus > di_minus: scores['ADX'] = 10
    elif adx > 25:                        scores['ADX'] = 5
    else:                                 scores['ADX'] = 0
 
    # Volume Profile (15 pts)
    if vp:
        status = vp['breakout_status']
        if status == 'BREAKOUT_UP':    scores['VP'] = 15
        elif status == 'AT_RESISTANCE':scores['VP'] = 10
        elif status == 'INSIDE':       scores['VP'] = 5
        else:                          scores['VP'] = 0
    else:
        scores['VP'] = 5
 
    return sum(scores.values()), scores

In [18]:
# ── CELL 8: REVERSAL SCORER ────────────────────────────────
def score_reversal(df, vp, lookback=20):
    scores = {}
    if len(df) < lookback + 10:
        return 0, {}
 
    # RSI level (20 pts)
    latest_rsi = df['RSI'].iloc[-1]
    if latest_rsi < 35:    scores['RSI_Level'] = 20
    elif latest_rsi < 45:  scores['RSI_Level'] = 10
    else:                  scores['RSI_Level'] = 0
 
    # RSI divergence (15 pts)
    rsi_5d    = df['RSI'].tail(5).min()
    rsi_20d   = df['RSI'].tail(20).min()
    price_5d  = df['Close'].tail(5).min()
    price_20d = df['Close'].tail(20).min()
    if price_5d <= price_20d and rsi_5d > rsi_20d:
        scores['RSI_Divergence'] = 15
    else:
        scores['RSI_Divergence'] = 0
 
    # MACD histogram (20 pts)
    latest_hist = df['MACD_Hist'].iloc[-1]
    prev_hist   = df['MACD_Hist'].iloc[-5]
    if latest_hist > 0 and prev_hist < 0:
        scores['MACD_Cross']  = 20
        scores['MACD_Rising'] = 0
    elif latest_hist > prev_hist and latest_hist < 0:
        scores['MACD_Cross']  = 0
        scores['MACD_Rising'] = 15
    else:
        scores['MACD_Cross']  = 0
        scores['MACD_Rising'] = 0
 
    # MACD divergence (15 pts)
    macd_5d  = df['MACD_Hist'].tail(5).min()
    macd_20d = df['MACD_Hist'].tail(20).min()
    if price_5d <= price_20d and macd_5d > macd_20d:
        scores['MACD_Divergence'] = 15
    else:
        scores['MACD_Divergence'] = 0
 
    # ADX declining (15 pts)
    adx_latest = df['ADX'].iloc[-1]
    adx_10d    = df['ADX'].iloc[-10]
    if adx_latest < adx_10d and adx_latest > 20:  scores['ADX'] = 15
    elif adx_latest < adx_10d:                    scores['ADX'] = 8
    else:                                          scores['ADX'] = 0
 
    # Volume drying up (15 pts)
    vol_ratio = df['Vol_Ratio'].iloc[-1]
    if vol_ratio < 0.6:    scores['Volume'] = 15
    elif vol_ratio < 0.8:  scores['Volume'] = 8
    else:                  scores['Volume'] = 0
 
    return min(sum(scores.values()), 100), scores

In [19]:
# ── CELL 9: CONSOLIDATION DETECTION (300+ days) ────────────
def detect_consolidation_dynamic(df,
                                  min_days=300,
                                  max_range_pct=50,
                                  max_total_drift=10,
                                  inside_pct_required=0.82):
    close = df['Close']
    high  = df['High']
    low   = df['Low']
    n     = len(df)
 
    if n < min_days + 5:
        return {'found': False, 'is_valid': False,
                'reason': 'Insufficient data'}
 
    best_result = None
 
    for lookback in range(min_days, min(800, n - 5)):
        start_idx  = n - lookback
        segment    = df.iloc[start_idx:n]
        seg_high   = segment['High']
        seg_low    = segment['Low']
        seg_close  = segment['Close']
 
        range_high = seg_high.quantile(0.95)
        range_low  = seg_low.quantile(0.05)
        range_pct  = (range_high - range_low) / range_low * 100
        rng        = range_high - range_low
 
        if range_pct > max_range_pct or rng == 0:
            continue
 
        buffer      = 0.03
        bars_inside = (
            (seg_high <= range_high * (1 + buffer)) &
            (seg_low  >= range_low  * (1 - buffer))
        ).sum()
        pct_inside  = bars_inside / len(segment)
 
        if pct_inside < inside_pct_required:
            continue
 
        x          = np.arange(len(segment))
        hi_slope   = np.polyfit(x, seg_high.values,  1)[0]
        hi_norm    = hi_slope / seg_high.mean()  * 100
        lo_slope   = np.polyfit(x, seg_low.values,   1)[0]
        lo_norm    = lo_slope / seg_low.mean()   * 100
        cl_slope   = np.polyfit(x, seg_close.values, 1)[0]
        cl_norm    = abs(cl_slope) / seg_close.mean() * 100
 
        hi_drift   = abs(hi_norm) * lookback
        lo_drift   = abs(lo_norm) * lookback
        cl_drift   = cl_norm      * lookback
 
        if hi_drift > max_total_drift or \
           lo_drift > max_total_drift or \
           cl_drift > max_total_drift:
            continue
 
        price_start  = seg_close.iloc[0]
        price_end    = seg_close.iloc[-1]
        price_change = (price_end - price_start) / price_start * 100
 
        if abs(price_change) > 25:
            continue
 
        end_vs_range = (price_end - range_low) / rng * 100
        if end_vs_range < -10:
            continue
 
        close_arr     = seg_close.values
        peak_idx      = np.argmax(close_arr)
        peak_pos      = peak_idx / lookback * 100
        peak_price    = close_arr.max()
        peak_vs_range = (peak_price  - range_low) / rng * 100
        start_vs_rng  = (price_start - range_low) / rng * 100
 
        is_mountain = (
            peak_pos      > 20  and
            peak_pos      < 80  and
            start_vs_rng  < 35  and
            end_vs_range  < 35  and
            peak_vs_range > 90
        )
        if is_mountain:
            continue
 
        pre_start   = max(0, start_idx - 120)
        pre_segment = df.iloc[pre_start:start_idx]
        pre_close   = pre_segment['Close'].values
 
        if len(pre_close) > 5:
            pre_x     = np.arange(len(pre_close))
            pre_slope = np.polyfit(pre_x, pre_close, 1)[0]
            pre_norm  = pre_slope / pre_close.mean() * 100
            pre_limit = -0.15 if lookback < 365 else -0.60
            if pre_norm < pre_limit:
                continue
            pre_high   = pre_segment['High'].max()
            drop_pct   = (range_high - pre_high) / pre_high * 100
            drop_limit = -20 if lookback < 365 else -60
            if drop_pct < drop_limit:
                continue
 
        if lookback > (best_result['consol_days'] if best_result else 0):
            best_result = {
                'consol_days' : lookback,
                'range_high'  : round(range_high, 2),
                'range_low'   : round(range_low,  2),
                'range_pct'   : round(range_pct,  2),
                'hi_drift'    : round(hi_drift,   2),
                'lo_drift'    : round(lo_drift,   2),
                'cl_drift'    : round(cl_drift,   2),
                'pct_inside'  : round(pct_inside, 3),
                'price_change': round(price_change, 2),
                'end_vs_range': round(end_vs_range, 1),
                'is_sideways' : True,
                'start_idx'   : start_idx,
            }
 
    if best_result is None:
        return {'found': False, 'is_valid': False,
                'consol_days': 0,
                'reason': 'No consolidation found'}
 
    consol_days   = best_result['consol_days']
    range_high    = best_result['range_high']
    range_low     = best_result['range_low']
    current_price = close.iloc[-1]
 
    consol_df   = df.iloc[best_result['start_idx']:n]
    res_touches = int((consol_df['High'] >= range_high * 0.98).sum())
    sup_touches = int((consol_df['Low']  <= range_low  * 1.02).sum())
 
    pct_above   = (current_price - range_high) / range_high * 100
    is_breaking = current_price >= range_high * 0.98
    vol_ratio   = round(df['Vol_Ratio'].tail(5).max(), 2)
 
    vp = calculate_volume_profile(
        consol_df, lookback_days=consol_days, bins=20
    )
    has_bell = vp['is_bell'] if vp else False
 
    if consol_days < 365:   duration_label = f'{consol_days}d (under 1 year)'
    elif consol_days < 500: duration_label = f'{consol_days}d (1-1.5 years)'
    elif consol_days < 730: duration_label = f'{consol_days}d (1.5-2 years)'
    else:                   duration_label = f'{consol_days}d (2+ years)'
 
    is_valid = res_touches >= 2 and sup_touches >= 2
 
    return {
        'found'               : True,
        'is_valid'            : is_valid,
        'consol_days'         : consol_days,
        'duration_label'      : duration_label,
        'range_high'          : range_high,
        'range_low'           : range_low,
        'range_pct'           : best_result['range_pct'],
        'hi_drift'            : best_result['hi_drift'],
        'lo_drift'            : best_result['lo_drift'],
        'pct_inside'          : best_result['pct_inside'],
        'price_change'        : best_result['price_change'],
        'end_vs_range'        : best_result['end_vs_range'],
        'is_sideways'         : True,
        'resistance_touches'  : res_touches,
        'support_touches'     : sup_touches,
        'is_breaking_out'     : is_breaking,
        'pct_above'           : round(pct_above,  2),
        'breakout_volume'     : vol_ratio,
        'has_bell'            : has_bell,
        'current_price'       : round(current_price, 2),
        'reason'              : 'Valid' if is_valid else
                                f'R:{res_touches} S:{sup_touches}'
    }
 

In [20]:
# ── CELL 10: CONSOLIDATION INFO WRAPPER ────────────────────
def get_consolidation_info(df):
    result = detect_consolidation_dynamic(df)
 
    if not result.get('found') or not result.get('is_valid'):
        return {
            'consolidating'     : False,
            'consol_days'       : 0,
            'duration_label'    : 'No consolidation',
            'range_high'        : None,
            'range_low'         : None,
            'range_pct'         : None,
            'pct_to_breakout'   : None,
            'is_breaking_out'   : False,
            'breakout_volume'   : None,
            'has_bell'          : False,
            'resistance_touches': 0,
            'support_touches'   : 0,
            'current_price'     : df['Close'].iloc[-1],
        }
 
    return {
        'consolidating'     : True,
        'consol_days'       : result['consol_days'],
        'duration_label'    : result['duration_label'],
        'range_high'        : result['range_high'],
        'range_low'         : result['range_low'],
        'range_pct'         : result['range_pct'],
        'pct_to_breakout'   : result['pct_above'],
        'is_breaking_out'   : result['is_breaking_out'],
        'breakout_volume'   : result['breakout_volume'],
        'has_bell'          : result['has_bell'],
        'resistance_touches': result['resistance_touches'],
        'support_touches'   : result['support_touches'],
        'current_price'     : result['current_price'],
    }
 
 
# ── CELL 11: REASON FUNCTION ────────────────────────────────
def get_reason(row):
    rsi    = float(row['RSI'])
    adx    = float(row['ADX'])
    setup  = str(row['Best Setup'])
    hist   = float(row['MACD Hist'])
    di_p   = float(row.get('DI+', 0))
    di_m   = float(row.get('DI-', 0))
    ema50  = float(row.get('EMA50',  0))
    ema200 = float(row.get('EMA200', 0))
    price  = float(row.get('Current Price', 0))
 
    uptrend   = (adx > 25 and di_p > di_m and
                 price > ema50 and price > ema200)
    bounce    = (price > ema50 and price < ema200)
    downtrend = (price < ema200) or (adx > 25 and di_m > di_p)
 
    if setup == 'Momentum':
        if uptrend:
            return f"Uptrend confirmed, RSI {rsi:.0f}, ADX {adx:.0f}"
        elif bounce:
            return f"Short bounce, below EMA200"
        elif price < ema200 and rsi > 50:
            return f"RSI {rsi:.0f} bounce in downtrend"
        elif rsi >= 50:
            return f"Momentum building, RSI {rsi:.0f}"
        return f"Momentum setup, RSI {rsi:.0f}"
 
    elif setup == 'Reversal':
        if rsi < 35 and hist > 0:
            return f"Oversold RSI {rsi:.0f}, MACD turning up"
        elif rsi < 35:
            return f"Oversold RSI {rsi:.0f}, watch for turn"
        elif hist > 0 and downtrend:
            return f"Downtrend weakening, MACD improving"
        return f"Divergence forming, RSI {rsi:.0f}"
 
    else:
        if downtrend and adx > 40:
            return f"Strong downtrend ADX {adx:.0f}, wait"
        elif downtrend and adx > 25:
            return f"Downtrend ADX {adx:.0f}, no setup yet"
        elif bounce:
            return f"Bounce in downtrend, below EMA200"
        elif rsi < 25:
            return f"Deeply oversold RSI {rsi:.0f}, too early"
        elif rsi < 40:
            return f"Weak RSI {rsi:.0f}, no momentum"
        elif rsi > 60 and not uptrend:
            return f"RSI {rsi:.0f} bounce, below EMA200"
        elif rsi > 60 and uptrend:
            return f"Uptrend, fund score low"
        else:
            return f"RSI {rsi:.0f}, setup not confirmed"

In [21]:
# ── CELL 12: GENERATE TECHNICAL REPORT ─────────────────────
print("Generating technical report for all 35 stocks...")
 
tech_reports = []
 
for _, fund_row in fundamental_df.iterrows():
    symbol     = fund_row['Symbol']
    fund_score = fund_row['Final Score']
    sector     = fund_row['Sector']
 
    if symbol not in indicator_data:
        continue
 
    df       = indicator_data[symbol]
    latest   = df.iloc[-1]
 
    close    = latest['Close']
    rsi      = round(latest['RSI'],       2)
    adx      = round(latest['ADX'],       2)
    hist     = round(latest['MACD_Hist'], 2)
    vol_r    = round(latest['Vol_Ratio'], 2)
    ema20    = round(latest['EMA20'],     2)
    ema50    = round(latest['EMA50'],     2)
    ema200   = round(latest['EMA200'],    2)
    di_plus  = round(latest['DI_Plus'],   2)
    di_minus = round(latest['DI_Minus'],  2)
 
    vp           = calculate_volume_profile(df)
    mom_score, _ = score_momentum(df, vp)
    rev_score, _ = score_reversal(df, vp)
    consol_info  = get_consolidation_info(df)
 
    if mom_score >= rev_score and mom_score >= 50:
        best_setup = 'Momentum'
        tech_score = mom_score
    elif rev_score >= mom_score and rev_score >= 50:
        best_setup = 'Reversal'
        tech_score = rev_score
    else:
        best_setup = 'Watching'
        tech_score = max(mom_score, rev_score)
 
    in_consol       = consol_info['consolidating']
    consol_days     = consol_info['consol_days']
    consol_label    = consol_info['duration_label']
    pct_to_breakout = consol_info['pct_to_breakout'] or 0
    consol_volume   = consol_info['breakout_volume']  or 0
 
    # Tier classification
    if fund_score >= 60 and tech_score >= 65 and best_setup != 'Watching':
        if best_setup == 'Momentum':
            tier = 'TIER 1 — BUY NOW (Momentum)'
        else:
            tier = 'TIER 1 — BUY NOW (Reversal)'
    elif fund_score >= 60 and in_consol and pct_to_breakout > -5:
        tier = 'TIER 1 — BREAKOUT IMMINENT'
    elif fund_score >= 60 and tech_score >= 40 and best_setup != 'Watching':
        tier = 'TIER 2 — WATCHLIST'
    elif fund_score >= 60 and in_consol:
        tier = 'TIER 2 — BASE BUILDING'
    else:
        tier = 'TIER 3 — WAITING'
 
    tech_reports.append({
        'Symbol'          : symbol,
        'Sector'          : sector,
        'Fund Score'      : fund_score,
        'Current Price'   : round(close,   2),
        'RSI'             : rsi,
        'ADX'             : adx,
        'MACD Hist'       : hist,
        'Vol Ratio'       : vol_r,
        'EMA20'           : ema20,
        'EMA50'           : ema50,
        'EMA200'          : ema200,
        'DI+'             : di_plus,
        'DI-'             : di_minus,
        'Momentum Score'  : mom_score,
        'Reversal Score'  : rev_score,
        'Best Setup'      : best_setup,
        'Tech Score'      : tech_score,
        'Tier'            : tier,
        'In Consolidation': in_consol,
        'Consol Days'     : consol_days,
        'Consol Label'    : consol_label,
        'Pct to Breakout' : round(pct_to_breakout, 2),
        'Consol Volume'   : consol_volume,
    })
    print(f"  {symbol:15} {tier}")
 
tech_report_df = pd.DataFrame(tech_reports)
 
# Add sector scores
fund_df = pd.read_csv('../data/fundamental_scores.csv')
fund_df['Sector Score'] = 0.0
for sector in fund_df['Sector'].unique():
    mask      = fund_df['Sector'] == sector
    max_score = fund_df[mask]['Final Score'].max()
    fund_df.loc[mask, 'Sector Score'] = (
        fund_df[mask]['Final Score'] / max_score * 10
    ).round(1)
 
tech_report_df = tech_report_df.merge(
    fund_df[['Symbol', 'Sector Score']],
    on='Symbol', how='left'
)
 
tech_report_df.to_csv('../data/technical_report.csv', index=False)
print(f"\nReport saved: {len(tech_report_df)} stocks")
print(f"\nTier distribution:")
print(tech_report_df['Tier'].value_counts().to_string())
 
 
# ── CELL 13: CONSOLIDATION REPORT (300+ days) ──────────────
print("=== CONSOLIDATION STATUS REPORT (300+ DAYS) ===\n")
print(f"{'Symbol':15} {'Fund':6} {'Days':6} "
      f"{'Range%':7} {'To Break':9} {'Vol':5} {'Bell':5} {'Duration'}")
print("─" * 75)
 
consol_info_list = []
 
for _, fund_row in fundamental_df.iterrows():
    symbol = fund_row['Symbol']
    if symbol not in indicator_data:
        continue
 
    df     = indicator_data[symbol]
    result = detect_consolidation_dynamic(df)
 
    if result['found'] and result['is_valid']:
        consol_info_list.append({
            'Symbol'         : symbol,
            'Fund Score'     : fund_row['Final Score'],
            'Sector'         : fund_row['Sector'],
            'consol_days'    : result['consol_days'],
            'duration_label' : result['duration_label'],
            'range_pct'      : result['range_pct'],
            'range_high'     : result['range_high'],
            'range_low'      : result['range_low'],
            'pct_to_breakout': result['pct_above'],
            'is_breaking_out': result['is_breaking_out'],
            'breakout_volume': result['breakout_volume'],
            'has_bell'       : result['has_bell'],
            'current_price'  : result['current_price'],
        })
 
        bell = 'Y' if result['has_bell'] else 'N'
        brk  = 'BREAKOUT' if result['is_breaking_out'] else ''
        print(f"{symbol:15} "
              f"{fund_row['Final Score']:5.1f}  "
              f"{result['consol_days']:5}d  "
              f"{result['range_pct']:6.1f}%  "
              f"{result['pct_above']:+7.1f}%  "
              f"{result['breakout_volume']:.1f}x  "
              f"{bell}  "
              f"{result['duration_label']} {brk}")
    else:
        print(f"{symbol:15} Not consolidating")
 
consol_df = pd.DataFrame(consol_info_list).sort_values(
    'consol_days', ascending=False
).reset_index(drop=True)
consol_df.to_csv('../data/consolidation_info.csv', index=False)
 
print(f"\nStocks in 300+ day consolidation: {len(consol_info_list)}/35")
print("Saved consolidation_info.csv")
 

Generating technical report for all 35 stocks...
  TCS             TIER 3 — WAITING
  RELIANCE        TIER 3 — WAITING
  WIPRO           TIER 3 — WAITING
  PERSISTENT      TIER 1 — BUY NOW (Reversal)
  LTTS            TIER 1 — BUY NOW (Reversal)
  HAPPSTMNDS      TIER 2 — WATCHLIST
  CHOLAFIN        TIER 3 — WAITING
  MUTHOOTFIN      TIER 3 — WAITING
  MANAPPURAM      TIER 3 — WAITING
  CREDITACC       TIER 3 — WAITING
  GARFIBRES       TIER 2 — WATCHLIST
  SUPRAJIT        TIER 3 — WAITING
  HIMATSEIDE      TIER 3 — WAITING
  HAL             TIER 2 — BASE BUILDING
  BEL             TIER 3 — WAITING
  MIDHANI         TIER 3 — WAITING
  PARAS           TIER 3 — WAITING
  VINATIORGA      TIER 3 — WAITING
  FINEORG         TIER 2 — WATCHLIST
  GALAXYSURF      TIER 3 — WAITING
  AAVAS           TIER 3 — WAITING
  SUNPHARMA       TIER 1 — BUY NOW (Momentum)
  DIVISLAB        TIER 3 — WAITING
  APLLTD          TIER 3 — WAITING
  GRANULES        TIER 2 — BASE BUILDING
  IPCALAB         TIER 3 

In [35]:
# ============================================================
# COMPLETE WEEKLY REPORT — FINAL VERSION
# ============================================================

import yfinance as yf

# ── STEP 1: Fetch Market Cap ─────────────────────────────────
print("Fetching market cap data...")
mcap_data = {}
for symbol in fundamental_df['Symbol'].tolist():
    try:
        ticker  = yf.Ticker(f"{symbol}.NS")
        info    = ticker.info
        mcap    = info.get('marketCap', 0)
        mcap_data[symbol] = mcap / 1e7
    except:
        mcap_data[symbol] = 0
print(f"Done: {len(mcap_data)} stocks")

# ── STEP 2: Classify Market Cap ──────────────────────────────
def classify_mcap(mcap_cr):
    if mcap_cr >= 100000:  return 'Large Cap'
    elif mcap_cr >= 25000: return 'Mini Large Cap'
    elif mcap_cr >= 5000:  return 'Mid Cap'
    else:                  return 'Small Cap'

# ── STEP 3: Load fund_df and calculate all scores ────────────
fund_df = pd.read_csv('../data/fundamental_scores.csv')

fund_df['Market Cap Cr'] = fund_df['Symbol'].map(mcap_data)
fund_df['Cap Category']  = fund_df['Market Cap Cr'].apply(classify_mcap)

# Sector Score = stock / best in same sector * 10
fund_df['Sector Score'] = 0.0
for sector in fund_df['Sector'].unique():
    mask      = fund_df['Sector'] == sector
    max_score = fund_df[mask]['Final Score'].max()
    if max_score > 0:
        fund_df.loc[mask, 'Sector Score'] = (
            fund_df[mask]['Final Score'] / max_score * 10
        ).round(1)

# Cap Score = stock / best in same sector + same cap * 10
fund_df['Cap Score'] = 0.0
for sector in fund_df['Sector'].unique():
    for cap in ['Large Cap', 'Mini Large Cap', 'Mid Cap', 'Small Cap']:
        mask   = ((fund_df['Sector'] == sector) &
                  (fund_df['Cap Category'] == cap))
        subset = fund_df[mask]
        if len(subset) == 0:
            continue
        max_score = subset['Final Score'].max()
        if max_score > 0:
            fund_df.loc[mask, 'Cap Score'] = (
                subset['Final Score'] / max_score * 10
            ).round(1)

# ── STEP 4: Merge into tech_report_df ────────────────────────
tech_report_df = tech_report_df.drop(
    columns=['Sector Score', 'Cap Score',
             'Market Cap Cr', 'Cap Category'],
    errors='ignore'
).merge(
    fund_df[['Symbol', 'Sector Score', 'Cap Score',
             'Market Cap Cr', 'Cap Category']],
    on='Symbol', how='left'
)

# ── STEP 5: Report Functions ─────────────────────────────────
cap_order = ['Large Cap', 'Mini Large Cap', 'Mid Cap', 'Small Cap']
cap_short = {
    'Large Cap'      : 'L',
    'Mini Large Cap' : 'ML',
    'Mid Cap'        : 'M',
    'Small Cap'      : 'S'
}

def print_tier1_cap(df_tier):
    for cap in cap_order:
        cap_stocks = df_tier[
            df_tier['Cap Category'] == cap
        ].sort_values('Fund Score', ascending=False)
        if len(cap_stocks) == 0:
            continue
        print(f"\n  [{cap_short[cap]}] {cap}")
        print(f"  {'─'*68}")
        for _, row in cap_stocks.iterrows():
            consol = (f"\n    Base    : {row['Consol Label']} "
                      f"| {row['Pct to Breakout']:+.1f}% to breakout"
                      if row['In Consolidation'] else "")
            print(f"\n    {row['Symbol']}  "
                  f"({row['Sector']})  "
                  f"MCap Rs{row['Market Cap Cr']:,.0f}Cr")
            print(f"    Setup   : {row['Best Setup']:10}  "
                  f"Tech {row['Tech Score']:3}/100")
            print(f"    Scores  : Fund {row['Fund Score']:4.1f}/100  "
                  f"Sector Rank {row['Sector Score']:4.1f}/10  "
                  f"Cap Rank {row['Cap Score']:4.1f}/10")
            print(f"    Price   : Rs{row['Current Price']:,.2f}  "
                  f"RSI {row['RSI']:.0f}  "
                  f"ADX {row['ADX']:.0f}  "
                  f"MACD {row['MACD Hist']:+.2f}")
            if consol:
                print(consol)
            print(f"    Reason  : {get_reason(row)}")


def print_tier2_cap(df_tier):
    for cap in cap_order:
        cap_stocks = df_tier[
            df_tier['Cap Category'] == cap
        ].sort_values('Fund Score', ascending=False)
        if len(cap_stocks) == 0:
            continue
        print(f"\n  [{cap_short[cap]}] {cap}")
        print(f"  {'Symbol':12} {'Fund':5} {'SecRank':8} {'CapRank':8} "
              f"{'Setup':10} {'Tech':5} {'RSI':5} {'ADX':5}  Reason")
        print(f"  {'─'*78}")
        for _, row in cap_stocks.iterrows():
            consol = f" [{row['Consol Days']}d]" \
                if row['In Consolidation'] else ""
            print(f"  {row['Symbol']:12} "
                  f"{row['Fund Score']:4.1f}  "
                  f"{row['Sector Score']:6.1f}/10  "
                  f"{row['Cap Score']:6.1f}/10  "
                  f"{row['Best Setup']:10} "
                  f"{row['Tech Score']:4}  "
                  f"{row['RSI']:4.0f}  "
                  f"{row['ADX']:4.0f}  "
                  f"{get_reason(row)}{consol}")


def print_tier2b_cap(df_tier):
    for cap in cap_order:
        cap_stocks = df_tier[
            df_tier['Cap Category'] == cap
        ].sort_values('Fund Score', ascending=False)
        if len(cap_stocks) == 0:
            continue
        print(f"\n  [{cap_short[cap]}] {cap}")
        print(f"  {'Symbol':12} {'Fund':5} {'SecRank':8} {'CapRank':8} "
              f"{'Days':6} {'To Break':9} {'Vol':5}  Reason")
        print(f"  {'─'*78}")
        for _, row in cap_stocks.iterrows():
            pct = row['Pct to Breakout']
            print(f"  {row['Symbol']:12} "
                  f"{row['Fund Score']:4.1f}  "
                  f"{row['Sector Score']:6.1f}/10  "
                  f"{row['Cap Score']:6.1f}/10  "
                  f"{row['Consol Days']:5}d  "
                  f"{pct:+8.1f}%  "
                  f"{row['Consol Volume']:.1f}x  "
                  f"{get_reason(row)}")


def print_tier3_cap(df_tier):
    for cap in cap_order:
        cap_stocks = df_tier[
            df_tier['Cap Category'] == cap
        ].sort_values('Fund Score', ascending=False)
        if len(cap_stocks) == 0:
            continue
        print(f"\n  [{cap_short[cap]}] {cap}")
        print(f"  {'Symbol':12} {'Fund':5} {'SecRank':8} {'CapRank':8} "
              f"{'RSI':5} {'ADX':5}  Why waiting")
        print(f"  {'─'*72}")
        for _, row in cap_stocks.iterrows():
            consol = f" [{row['Consol Days']}d base]" \
                if row['In Consolidation'] else ""
            print(f"  {row['Symbol']:12} "
                  f"{row['Fund Score']:4.1f}  "
                  f"{row['Sector Score']:6.1f}/10  "
                  f"{row['Cap Score']:6.1f}/10  "
                  f"{row['RSI']:4.0f}  "
                  f"{row['ADX']:4.0f}  "
                  f"{get_reason(row)}{consol}")


# ── STEP 6: Print Full Report ─────────────────────────────────
print("=" * 78)
print("  AI STOCK SCREENER — WEEKLY REPORT")
print(f"  {pd.Timestamp.now().strftime('%d %B %Y')}")
print("=" * 78)
print(f"\n  SecRank = rank vs same sector  |  "
      f"CapRank = rank vs same sector + same cap")

# TIER 1
tier1 = tech_report_df[
    tech_report_df['Tier'].str.contains('TIER 1')
].sort_values('Fund Score', ascending=False)

print(f"\n{'─'*78}")
print(f"  TIER 1 — ACT NOW  ({len(tier1)} stocks)")
print(f"  Fundamentally strong + Technical setup confirmed")
print(f"{'─'*78}")
print_tier1_cap(tier1)

# TIER 2A
tier2_watch = tech_report_df[
    tech_report_df['Tier'] == 'TIER 2 — WATCHLIST'
].sort_values('Fund Score', ascending=False)

if len(tier2_watch) > 0:
    print(f"\n{'─'*78}")
    print(f"  TIER 2A — WATCHLIST  ({len(tier2_watch)} stocks)")
    print(f"  Setup forming — wait for confirmation before entering")
    print(f"{'─'*78}")
    print_tier2_cap(tier2_watch)

# TIER 2B
tier2_base = tech_report_df[
    tech_report_df['Tier'] == 'TIER 2 — BASE BUILDING'
].sort_values('Fund Score', ascending=False)

if len(tier2_base) > 0:
    print(f"\n{'─'*78}")
    print(f"  TIER 2B — BASE BUILDING  ({len(tier2_base)} stocks)")
    print(f"  Long consolidation base — watch for volume breakout")
    print(f"{'─'*78}")
    print_tier2b_cap(tier2_base)

# TIER 3
tier3 = tech_report_df[
    tech_report_df['Tier'] == 'TIER 3 — WAITING'
].sort_values('Fund Score', ascending=False)

print(f"\n{'─'*78}")
print(f"  TIER 3 — WAITING  ({len(tier3)} stocks)")
print(f"  Good businesses — technical setup not ready yet")
print(f"{'─'*78}")
print_tier3_cap(tier3)

# LONG BASE WATCH
consol_stocks = tech_report_df[
    tech_report_df['In Consolidation'] == True
].sort_values('Consol Days', ascending=False)

print(f"\n{'─'*78}")
print(f"  LONG BASE WATCH  ({len(consol_stocks)} stocks, 300+ days)")
print(f"  Watch for breakout above range high with 2x+ volume")
print(f"{'─'*78}")
print(f"  {'Symbol':12} {'Category':14} {'Fund':5} "
      f"{'SecRank':8} {'CapRank':8} "
      f"{'Days':6} {'To Break':9} {'Vol':5}  Status")
print(f"  {'─'*78}")

for _, row in consol_stocks.iterrows():
    pct = row['Pct to Breakout']
    if pct > -5:    status = "NEAR BREAKOUT"
    elif pct > -15: status = "Approaching"
    else:           status = "Early stage"
    print(f"  {row['Symbol']:12} "
          f"{row['Cap Category']:14} "
          f"{row['Fund Score']:4.1f}  "
          f"{row['Sector Score']:6.1f}/10  "
          f"{row['Cap Score']:6.1f}/10  "
          f"{row['Consol Days']:5}d  "
          f"{pct:+8.1f}%  "
          f"{row['Consol Volume']:.1f}x  "
          f"{status}")

print(f"\n{'─'*78}")
print(f"  L=Large Cap  ML=Mini Large Cap  M=Mid Cap  S=Small Cap")
print(f"  SecRank = rank vs same sector stocks")
print(f"  CapRank = rank vs same sector + same cap stocks")
print(f"{'─'*78}")

# ── STEP 7: Save updated report ──────────────────────────────
tech_report_df.to_csv('../data/technical_report.csv', index=False)
print("\nSaved: data/technical_report.csv ✅")

Fetching market cap data...
Done: 35 stocks
  AI STOCK SCREENER — WEEKLY REPORT
  15 March 2026

  SecRank = rank vs same sector  |  CapRank = rank vs same sector + same cap

──────────────────────────────────────────────────────────────────────────────
  TIER 1 — ACT NOW  (4 stocks)
  Fundamentally strong + Technical setup confirmed
──────────────────────────────────────────────────────────────────────────────

  [L] Large Cap
  ────────────────────────────────────────────────────────────────────

    SUNPHARMA  (Healthcare)  MCap Rs432,264Cr
    Setup   : Momentum    Tech 100/100
    Scores  : Fund 65.0/100  Sector Rank 10.0/10  Cap Rank 10.0/10
    Price   : Rs1,801.60  RSI 61  ADX 40  MACD +5.11

    Base    : 520d (1.5-2 years) | -2.7% to breakout
    Reason  : Uptrend confirmed, RSI 61, ADX 40

  [ML] Mini Large Cap
  ────────────────────────────────────────────────────────────────────

    PERSISTENT  (Information Technology)  MCap Rs72,535Cr
    Setup   : Reversal    Tech 100/1

In [36]:
# ── CAP SCORE = sector + cap combination ────────────────
fund_df['Cap Score'] = 0.0

for sector in fund_df['Sector'].unique():
    for cap in ['Large Cap', 'Mini Large Cap', 'Mid Cap', 'Small Cap']:
        mask   = (fund_df['Sector'] == sector) & \
                 (fund_df['Cap Category'] == cap)
        subset = fund_df[mask]

        if len(subset) == 0:
            continue

        max_score = subset['Final Score'].max()
        if max_score > 0:
            fund_df.loc[mask, 'Cap Score'] = (
                subset['Final Score'] / max_score * 10
            ).round(1)

# Verify
print("=== CAP SCORE (Sector + Cap combination) ===\n")
for cap in ['Large Cap', 'Mini Large Cap', 'Mid Cap', 'Small Cap']:
    cap_stocks = fund_df[
        fund_df['Cap Category'] == cap
    ].sort_values(['Sector', 'Final Score'], ascending=[True, False])

    if len(cap_stocks) == 0:
        continue

    print(f"\n  {cap}")
    print(f"  {'Symbol':15} {'Fund':6} {'Sector':8} "
          f"{'Cap':6}  {'Sector Name'}")
    print(f"  {'─'*60}")

    for _, row in cap_stocks.iterrows():
        print(f"  {row['Symbol']:15} "
              f"{row['Final Score']:5.1f}  "
              f"{row['Sector Score']:5.1f}/10  "
              f"{row['Cap Score']:5.1f}/10  "
              f"{row['Sector']}")

=== CAP SCORE (Sector + Cap combination) ===


  Large Cap
  Symbol          Fund   Sector   Cap     Sector Name
  ────────────────────────────────────────────────────────────
  BEL              68.0   10.0/10   10.0/10  Capital Goods
  HAL              62.0    9.1/10    9.1/10  Capital Goods
  TITAN            69.0   10.0/10   10.0/10  Consumer Durables
  MUTHOOTFIN       73.4   10.0/10   10.0/10  Financial Services
  CHOLAFIN         70.7    9.6/10    9.6/10  Financial Services
  SUNPHARMA        65.0   10.0/10   10.0/10  Healthcare
  DIVISLAB         64.0    9.8/10    9.8/10  Healthcare
  TCS              64.0    8.0/10   10.0/10  Information Technology
  WIPRO            49.0    6.1/10    7.7/10  Information Technology
  RELIANCE         55.0   10.0/10   10.0/10  Oil, Gas & Consumable Fuels

  Mini Large Cap
  Symbol          Fund   Sector   Cap     Sector Name
  ────────────────────────────────────────────────────────────
  PIIND            68.0    9.3/10   10.0/10  Chemicals
  HA